In [18]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

def model_training_25d(x: np.ndarray, labels: np.ndarray):
    '''
    Takes in a series of ndarrays and trains a model using a 3D CNN architecture.
    '''
    # Input layer: Shape is (height, width, channels)
    input_tensor = layers.Input(shape=(1, 24, 224, 224, 1))

    ### First Convolution & MaxPooling
    net = layers.Conv3D(32, 3, padding='same', activation='relu')(input_tensor)
    net = layers.MaxPooling3D(pool_size=(1, 2, 2), padding='same')(net)

    ### Second Convolution & MaxPooling
    net = layers.Conv3D(64, 3, padding='same', activation='relu')(net)
    net = layers.MaxPooling3D(pool_size=(3, 3, 3), padding='same')(net)

    ### Third Convolution & MaxPooling
    net = layers.Conv3D(64, 3, padding='same', activation='relu')(net)
    net = layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same')(net)

    ### Flattening
    shared_feature = layers.TimeDistributed(layers.GlobalAveragePooling3D())(net)

    ### Fully Connected Layers
    net = layers.TimeDistributed(layers.Dense(20, activation='relu'))(shared_feature)
    net = layers.TimeDistributed(layers.Dense(20, activation='relu'))(shared_feature)

    ### Classification Layer: 12 independent binary classifications
    # One unit per condition (ACL, MCL, Meniscus, OA, Effusion, etc.)
    output_layer = layers.TimeDistributed(layers.Dense(12, activation='sigmoid'))(net)

    # Instantiate the Functional Model
    model = models.Model(inputs=input_tensor, outputs=output_layer)

    ### Compile the model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )

    # Train the model
    # Ensure x matches the input shape, e.g., (batch_size, 24, 224, 224, 1)
    # Ensure labels match shape: (batch_size, 12)
    model.fit(x, labels, epochs=1)

    return model


In [ ]:
# --- SIMULATED DATA FOR 2 PATIENTS ---
# Patient 1
p1_x = np.random.rand(1, 224, 224, 24, 1)
p1_y = np.array([[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]]) # 1 row of 12

# Patient 2
p2_x = np.random.rand(1, 224, 224, 24, 1)
p2_y = np.array([[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]]) # 1 row of 12

# --- COMBINING FOR THE TRAINING FUNCTION ---
# Stack patients along the batch axis (axis 0)
X_train = np.concatenate([p1_x, p2_x], axis=0)      # Shape: (2, 224, 224, 24)
Y_train = np.concatenate([p1_y, p2_y], axis=0)      # Shape: (2, 12)

# --- RUNNING THE MODEL ---
# Now x sizes (2) and y sizes (2) match perfectly!
trained_model = model_training_25d(X_train, Y_train)

ValueError: Input 0 of layer "functional_1" is incompatible with the layer: expected shape=(None, 1, 224, 224, 24), found shape=(None, 224, 224, 24)

In [7]:
Y_train

array([[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]])